In [1]:
import gensim.downloader as api

def run_gensim_tutorial():
    print("모델을 로딩 중입니다... (glove-wiki-gigaword-50)\n")
    model = api.load("glove-wiki-gigaword-50")
    word1, word2, word3 = 'cat', 'dog', 'car'
    
    sim_cat_dog = model.similarity(word1, word2)
    sim_cat_car = model.similarity(word1, word3)
    
    target_word = 'computer'
    similar_words = model.most_similar(target_word, topn=5)
    
    for word, score in similar_words:
        print(f" - {word} (유사도 점수: {score:.4f})")
    print()
    print("3. 단어 그룹 중 성격이 다른 단어(Outlier) 분류하기")
    word_group = ['apple', 'banana', 'orange', 'car']
    
    outlier = model.doesnt_match(word_group)
    print(f" - 단어 그룹: {word_group}")
    print(f" - 과일 그룹에 어울리지 않는 단어: '{outlier}'\n")

# 실행
if __name__ == "__main__":
    run_gensim_tutorial()

모델을 로딩 중입니다... (glove-wiki-gigaword-50)

 - computers (유사도 점수: 0.9165)
 - software (유사도 점수: 0.8815)
 - technology (유사도 점수: 0.8526)
 - electronic (유사도 점수: 0.8126)
 - internet (유사도 점수: 0.8060)

3. 단어 그룹 중 성격이 다른 단어(Outlier) 분류하기
 - 단어 그룹: ['apple', 'banana', 'orange', 'car']
 - 과일 그룹에 어울리지 않는 단어: 'car'



In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.????(text, allowed_special={"<|endoftext|>"})
# endregion
print(integers)

strings = tokenizer.????(integers)
# endregion
print(strings)


In [ ]:
import os
import requests

os.makedirs("datas", exist_ok=True)
if not os.path.exists("datas/the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "datas/the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)
with open("datas/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[????:context_size+?????]

print(f"x: {x}")
print(f"y:      {y}")

In [ ]:
### 중요: 이 코드는 GPT 모델 학습을 위한 데이터셋 클래스를 정의합니다

import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[???? : ???? + max_length]
            target_chunk = token_ids[???? + ???? : ???? + max_length + ????]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[????], self.target_ids[????]

In [ ]:
##중요 : 아래 함수는 DataLoader를 생성하는 헬퍼 함수입니다.
import tiktoken
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        ????,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    # endregion
    
    return dataloader

In [1]:
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(????, ????)

print(embedding_layer.weight)
token_embeddings = embedding_layer(????)
# endregion
print(token_embeddings)

SyntaxError: invalid syntax (3591370594.py, line 7)

In [2]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

pos_embedding_layer = torch.nn.Embedding(???, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(???))
print(pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

SyntaxError: invalid syntax (2642639326.py, line 17)

In [ ]:
### 중요: 이 코드는 인과적 어텐션 메커니즘을 구현한 PyTorch 모듈입니다.
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.W_query = nn.????(????, ????, bias=qkv_bias)
        self.W_key   = nn.????(????, ????, bias=qkv_bias)
        self.W_value = nn.????(????, ????, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask', 
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape 

        keys = self.????(????)    
        queries = self.?????(????)
        values = self.?????(????) 

        attn_scores = ???? @ ????.transpose(1, 2) 
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], 
            ????
        ) 
        attn_weights = torch.????(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ ????
        
        return context_vec

In [ ]:
### 중요: 이 코드는 인과적 어텐션 메커니즘을 구현한 PyTorch 모듈입니다.
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()


    def forward(self, x):
        
        return context_vec

In [3]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in = inputs.shape[1]   # 입력 차원 (d=3)
d_out = 2                # Q, K, V의 출력 차원 (d=2)

batch = torch.stack((inputs, inputs), dim=0)
# --- 실행 예시 ---
torch.manual_seed(123)
context_length = batch.shape[1] 

print("context_length:", context_length)
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ????(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

SyntaxError: invalid syntax (2664622917.py, line 22)

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = 

    def forward(self, x):
        return torch.cat()

torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = ???

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = ???
        values = ???
        queries = ???

        attn_scores = ??? # Dot product for each head

        mask_bool = self.???

        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = ???
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = ???
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

In [ ]:

#####################################
# Chapter 4: GPT 아키텍처 구성 요소
#####################################
class LayerNorm(nn.Module):
    """
    층 정규화 (Layer Normalization): 학습 안정성을 높임
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim)) # 학습 가능한 스케일 파라미터 (Gamma)
        self.shift = nn.Parameter(torch.zeros(emb_dim)) # 학습 가능한 시프트 파라미터 (Beta)

    def forward(self, x):
        mean = ???
        var = ???
        norm_x = (x - ????) / torch.sqrt(???? + self.eps)
        return self.???? * norm_x + self.????


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], ???? * cfg["emb_dim"]),
            ????(),
            nn.Linear(???? * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)



In [ ]:
##중요 : 아래는 GPT 모델 전체 구조를 정의하는 코드입니다

#####################################
# Chapter 4: GPT 아키텍처 조립
#####################################
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # region [어텐션 블록 (Residual Connection 적용)]
        shortcut = x
        x = self.norm1(x) # Pre-LayerNorm 방식
        x = self.????(x)   
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력을 더해줌 (기울기 소실 방지)
        
        shortcut = x
        x = self.norm2(x)
        x = self.????(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력 다시 더함

        return x
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["????"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["????"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["????"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(???)

        x = tok_embeds + ????
        x = self.drop_emb(x)
        x = self.????(x)
        x = self.final_norm(x)
        logits = self.????(x) # 각 단어에 대한 예측 점수 (Logits)
        
        return logits

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.????():
            logits = model(idx_cond)
        logits = logits[:, ????, :]

        idx_next = torch.????(logits, dim=-1, keepdim=True) 
        idx = torch.????((idx, idx_next), dim=1) 

    return idx

In [4]:
# Copyright (c) Sebastian Raschka under Apache License 2.0 (see LICENSE.txt).
# Source for "Build a Large Language Model From Scratch"
#   - https://www.manning.com/books/build-a-large-language-model-from-scratch
# Code: https://github.com/rasbt/LLMs-from-scratch

import matplotlib.pyplot as plt
import os
import requests
import torch
import tiktoken

In [ ]:
### 중요: lOSS를 계산합니다.
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.????(
        logits.????(0, 1), 
        target_batch.????()
    )
    return loss


In [ ]:
### 중요: 모델을 학습합니다.
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):

    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen = 0
    global_step = -1

    for epoch in range(num_epochs):
        model.????() 

        
        for input_batch, target_batch in train_loader:

            optimizer.????()

            loss = ????(input_batch, target_batch, model, device)

            loss.????() 

            optimizer.????() 

            tokens_seen += input_batch.numel() # 처리한 토큰 수 카운트
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen

In [5]:
def generate(model, idx, max_new_tokens, context_size, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, ????, :]
        idx_next = ???
        if idx_next == eos_id:
            break
        idx = ???
    return idx

SyntaxError: invalid syntax (3413835438.py, line 7)

In [ ]:
def text_to_token_ids(text, tokenizer):
    """
    사람이 읽는 텍스트 문자열을 모델이 이해하는 토큰 ID 텐서로 변환합니다.
    """
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).????(0) 
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    """
    모델이 뱉어낸 토큰 ID 텐서를 사람이 읽을 수 있는 텍스트로 변환합니다.
    """
    flat = token_ids.????(0) 
    return tokenizer.decode(flat.tolist())


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # 평가 시간을 단축하기 위해 전체 데이터를 다 보지 않고 일부만 볼 수 있게 설정
        num_batches = min(num_batches, len(data_loader))
        
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item() # 텐서에서 실수값(float)만 추출하여 누적
        else:
            break
    return total_loss / num_batches


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval() # [중요] 평가 모드 전환: Dropout이나 BatchNorm 등의 동작이 변경됨
    
    with torch.no_grad(): # [중요] 기울기(Gradient) 계산 끔 -> 메모리 절약 및 속도 향상
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
        
    model.train() # 평가가 끝나면 다시 훈련 모드로 복귀해야 함
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval() # 평가 모드
    context_size = model.pos_emb.weight.shape[0] # 모델이 처리 가능한 최대 길이
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
        decoded_text = token_ids_to_text(token_ids, tokenizer)
        
        # 출력이 너무 길어지면 보기 힘드므로 줄바꿈을 공백으로 변경
        print(decoded_text.replace("\n", " "))  
    
    model.train() # 훈련 모드 복귀



def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    """
    훈련 진행 상황(Loss 변화)을 시각화하는 함수입니다.
    X축을 'Epoch'와 '처리한 토큰 수' 두 가지 기준으로 보여줍니다.
    """
    fig, ax1 = plt.subplots()

    # 기본 X축: Epoch 기준
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")

    # 보조 X축 (상단): 처리한 토큰 수(Tokens seen) 기준
    ax2 = ax1.twiny() 
    ax2.plot(tokens_seen, train_losses, alpha=0) # 투명한 그래프로 축만 생성
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout() 
    # plt.show() # 주피터 노트북 환경이면 주석 해제


def main(gpt_config, settings):
    # 랜덤 시드 고정 (실험 재현성을 위해)
    torch.manual_seed(123)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    file_path = "datas/the-verdict.txt"
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    model = GPTModel(gpt_config)
    model.to(device) # 모델을 GPU 메모리로 올림

    optimizer = torch.optim.AdamW(
        model.????(), ????=settings["learning_rate"], weight_decay=settings["weight_decay"]
    )


    train_ratio = 0.90
    split_idx = int(train_ratio * len(text_data))

    # 훈련 데이터 로더: 순서를 섞음(Shuffle=True)
    train_loader = create_dataloader_v1(
        text_data[:split_idx],
        batch_size=settings["batch_size"],
        max_length=gpt_config["context_length"],
        stride=gpt_config["context_length"],
        drop_last=True,
        shuffle=True,
        num_workers=0
    )

    # 검증 데이터 로더: 순서를 섞지 않음(Shuffle=False) -> 평가는 일관되게
    val_loader = create_dataloader_v1(
        text_data[split_idx:],
        batch_size=settings["batch_size"],
        max_length=gpt_config["context_length"],
        stride=gpt_config["context_length"],
        drop_last=False,
        shuffle=False,
        num_workers=0
    )

    ##############################
    # 4. 훈련 시작
    ##############################
    tokenizer = tiktoken.get_encoding("gpt2") # BPE 토크나이저

    train_losses, val_losses, tokens_seen = train_model_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=settings["num_epochs"], eval_freq=5, eval_iter=1,
        start_context="Every effort moves you", tokenizer=tokenizer
    )

    return train_losses, val_losses, tokens_seen, model


In [6]:

# GPT-2 Small (124M) 모델 설정
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # 어휘 크기
    "context_length": 256,  # 훈련 속도를 위해 원본(1024)보다 줄임
    "emb_dim": 768,         # 임베딩 벡터 차원
    "n_heads": 12,          # 어텐션 헤드 개수
    "n_layers": 12,         # 레이어 깊이
    "drop_rate": 0.1,       # 과적합 방지 드롭아웃
    "qkv_bias": False       
}

# 학습 하이퍼파라미터
OTHER_SETTINGS = {
    "learning_rate": 5e-4, 
    "num_epochs": 10,       
    "batch_size": 2,        
    "weight_decay": 0.1     
}

###########################
# 메인 실행: 훈련
###########################
train_losses, val_losses, tokens_seen, model = main(GPT_CONFIG_124M, OTHER_SETTINGS)

###########################
# 훈련 결과 시각화 및 저장
###########################

# 에포크 축 생성 (0부터 num_epochs까지)
epochs_tensor = torch.linspace(0, OTHER_SETTINGS["num_epochs"], len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
plt.savefig("outputs/loss.pdf")

# 모델 저장 및 로드 테스트
torch.save(model.state_dict(), "outputs/model.pth")

# 디바이스 설정 (모델 로드 전에 정의 필요)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 다시 불러오기 테스트
model = GPTModel(GPT_CONFIG_124M) # 1. 빈 모델 생성 (현재 CPU에 있음)
model.load_state_dict(torch.load("outputs/model.pth", weights_only=True)) # 2. 가중치 로드
model.to(device) # <--- [핵심 수정] 3. 모델을 GPU로 이동시켜야 함

torch.manual_seed(123)
###########################
# 생성 테스트 (Inference)
###########################
tokenizer = tiktoken.get_encoding("gpt2")

# 훈련된 모델로 텍스트 생성 시도
# Temperature=1.4로 설정하여 다소 창의적/다양한 문장 생성 유도
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))

NameError: name 'main' is not defined

In [7]:
import math
import time
import requests
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt # 그래프 저장을 위해 추가
# -----------------------------------------------------------------------------
# 1. 설정 및 하이퍼파라미터
# -----------------------------------------------------------------------------
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
    "emb_dim": 768, "n_layers": 12, "n_heads": 12
}

BATCH_SIZE = 8
NUM_EPOCHS = 5
LEARNING_RATE = 8e-4
LORA_RANK = 16
LORA_ALPHA = 16


In [ ]:
###중요: LoRA 구현 부분
# -----------------------------------------------------------------------------
# 2. LoRA 클래스 및 함수 정의
# -----------------------------------------------------------------------------
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, ????))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        self.B = nn.Parameter(torch.zeros(????, out_dim))
        
        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.???? / self.rank) * (x @ self.???? @ self.????)
        return x

class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.????(x)

        
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.????):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_with_lora(module, rank, alpha)

In [ ]:
def main():
    # 시드 설정
    torch.manual_seed(123)
    
    # 장치 설정
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"사용 장치: {device}")

    # --- 데이터셋 준비 ---
    print("데이터셋 준비 중...")
    url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
    zip_path = "datas/sms_spam_collection.zip"
    extracted_path = "datas/sms_spam_collection"
    data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

    try:
        download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
    except (requests.exceptions.RequestException, TimeoutError):
        # 백업 URL 시도
        url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
        download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

    df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
    balanced_df = create_balanced_dataset(df)
    balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

    train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
    
    # 임시 파일 저장 (Dataset 클래스 사용을 위해)
    train_df.to_csv("datas/train.csv", index=None)
    validation_df.to_csv("datas/validation.csv", index=None)
    test_df.to_csv("datas/test.csv", index=None)

    tokenizer = tiktoken.get_encoding("gpt2")
    train_dataset = SpamDataset("datas/train.csv", max_length=None, tokenizer=tokenizer)
    val_dataset = SpamDataset("datas/validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
    test_dataset = SpamDataset("datas/test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)

    train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, drop_last=False)
    test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, drop_last=False)

    # --- 모델 준비 ---
    print("모델 로드 중...")

    file_name = "gpt2-small-124M.pth"
    model_path = f"./models/gpt2/{file_name}"
    download_model(file_name, model_path)

    # Policy Model (학습 대상)
    model = GPTModel(BASE_CONFIG)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.to(device)

    # 출력 헤드 교체 (분류용)
    num_classes = 2
    model.out_head = torch.nn.Linear(in_features=768, out_features=num_classes)
    
    # --- LoRA 적용 ---
    for param in model.parameters():
        param.requires_grad = ????

    ????(model, rank=LORA_RANK, alpha=LORA_ALPHA)
    
    model.to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"총 학습 가능 LoRA 파라미터 수: {total_params:,}")

    # --- 학습 실행 ---
    print("학습 시작...")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)
    
    start_time = time.time()
    # 반환값 캡처 (loss 및 accuracy 기록)
    train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=NUM_EPOCHS, eval_freq=50, eval_iter=5,
    )
    end_time = time.time()

In [ ]:
## Finetuning_Classification_without_LORA

def random_split(df, train_frac, validation_frac):
    """
    데이터셋을 학습(Train), 검증(Validation), 테스트(Test) 셋으로 분할하는 함수
    """
    # 전체 데이터 섞기
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # 분할 지점(인덱스) 계산
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # 데이터 분할
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df


class SpamDataset(Dataset):
    """
    PyTorch Dataset 클래스 정의.
    텍스트를 토큰화하고, 패딩(Padding) 처리를 수행합니다.
    """
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # 1. 텍스트 데이터를 토큰 ID 리스트로 변환 (Tokenization)
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        # 2. 최대 길이(max_length) 설정
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # max_length보다 긴 문장은 잘라냄 (Truncation)
            # TODO: 시퀀스를 최대 길이로 잘라낼 끝 인덱스를 채우세요.
            # 힌트: 설정된 최대 길이 `self.max_length`로 슬라이싱하면 됩니다.
            self.encoded_texts = [
                encoded_text[:????]
                for encoded_text in self.encoded_texts
            ]

        # 3. 패딩(Padding): 모든 시퀀스 길이를 max_length로 맞춤
        # GPT-2의 <|endoftext|> 토큰 ID인 50256을 패딩 값으로 사용
        # TODO: 패딩에 사용할 토큰 ID를 채우세요.
        # 힌트: 함수 파라미터로 전달된 `pad_token_id`를 사용합니다.
        self.encoded_texts = [
            encoded_text + [????] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        # TODO: 레이블을 가져올 DataFrame 컬럼명을 채우세요.
        # 힌트: 데이터셋의 정답 컬럼 이름은 'Label'입니다.
        label = self.data.iloc[index]["????"]
        # 텐서 형태로 반환 (입력 데이터, 정답 라벨)
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length


# -----------------------------------------------------------------------------
# 2. 평가 및 학습 관련 함수들
# -----------------------------------------------------------------------------

def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    """
    데이터 로더를 순회하며 모델의 정확도(Accuracy)를 계산
    """
    model.eval()  # 평가 모드 설정 (Dropout 비활성화 등)
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
        
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                # GPT 모델은 시퀀스를 출력하므로, 분류를 위해서는 '마지막 토큰'의 출력만 사용
                # logits shape: [batch_size, seq_len, vocab_size 또는 num_classes]
                logits = model(input_batch)[:, -1, :]  
            
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples


def calc_loss_batch(input_batch, target_batch, model, device):
    """
    단일 배치에 대한 손실(Loss) 계산
    """
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    # TODO: 분류에 사용할 시퀀스 내 위치 인덱스를 채우세요.
    # 힌트: GPT는 시퀀스의 마지막 토큰 출력만 분류 예측에 사용합니다.
    logits = model(input_batch)[:, ????, :]  
    # Cross Entropy Loss 계산
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    """
    현재 모델의 Train Loss와 Validation Loss를 평가
    """
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
num_workers = 0
batch_size = 8
torch.manual_seed(123)

# DataLoader 생성
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True, # 학습 데이터는 섞음
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

########################################
# 3단계: 사전 학습된 모델 로드 (Pre-trained Model Loading)
########################################

if test_mode:
    # 테스트 모드일 때는 아주 작은 더미 모델 생성
    BASE_CONFIG = {
        "vocab_size": 50257,
        "context_length": 120,
        "drop_rate": 0.0,
        "qkv_bias": False,
        "emb_dim": 12,
        "n_layers": 1,
        "n_heads": 2
    }
    model = GPTModel(BASE_CONFIG)
    model.eval()
    device = "cpu"

else:
    # 실제 GPT-2 Small (124M 파라미터) 모델 설정 및 가중치 로드
    CHOOSE_MODEL = "gpt2-small (124M)"
    
    BASE_CONFIG = {
        "vocab_size": 50257,     
        "context_length": 1024,  
        "drop_rate": 0.0,        
        "qkv_bias": True         
    }

    model_configs = {
        "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
        # 다른 모델 사이즈 설정들...
        "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
        "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
        "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
    }

    BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

    # 데이터셋의 길이가 모델의 컨텍스트 길이를 초과하는지 확인
    assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
        f"Dataset length {train_dataset.max_length} exceeds model's context "
        f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
        f"`max_length={BASE_CONFIG['context_length']}`"
    )

    model_name = "gpt2-small-124M.pth"
    model = load_gpt2_model(model_name, BASE_CONFIG)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for param in model.parameters():
    param.requires_grad = ????

torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=????)
# endregion

model.to(device)

for param in model.trf_blocks[????].parameters():
    param.requires_grad = ????

for param in model.final_norm.parameters():
    param.requires_grad = ????
# endregion

########################################
# 5단계: 모델 미세 조정 (Fine-tuning) 실행
########################################

start_time = time.time()
torch.manual_seed(123)

# 옵티마이저 설정 (학습 가능한 파라미터만 전달됨)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
# 학습 함수 호출
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
## Instruction Finetuning

class InstructionDataset(Dataset):
    """
    지시사항(Instruction) 데이터셋을 처리하는 PyTorch Dataset 클래스.
    입력 데이터(지시+입력)와 정답 데이터(응답)를 하나의 텍스트로 합쳐서 토큰화합니다.
    """
    def __init__(self, data, tokenizer):
        self.data = data

        # 텍스트 데이터 미리 토큰화 (Pre-tokenize)
        self.encoded_texts = []
        for entry in data:
            # 포맷팅 함수를 이용해 "지시사항 + 입력" 텍스트 생성
            instruction_plus_input = format_input(entry)

            # 정답(Response) 텍스트 생성
            response_text = f"\n\n### Response:\n{entry['output']}"
            
            # 모델은 이 전체 텍스트(질문+답변)를 보고 다음 토큰을 예측하도록 학습됨
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    """
    데이터 로더에서 배치를 만들 때 사용하는 커스텀 함수.
    가변 길이의 시퀀스를 배치의 최대 길이에 맞춰 패딩(padding)하고, 
    정답(target) 데이터에서 패딩 부분은 손실(loss) 계산에서 제외하도록 처리합니다.
    """
    # 배치 내에서 가장 긴 시퀀스 길이 계산 (패딩을 위해 +1 여유)
    batch_max_length = max(len(item)+1 for item in batch)

    # 입력(inputs)과 정답(targets) 리스트 준비
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # 문장 끝 토큰 <|endoftext|> 추가
        # 힌트: 패딩 전에 구분 토큰이 필요한 경우 pad_token_id를 추가합니다.
        new_item += [????]
        
        # 힌트: 모든 샘플 길이를 맞추기 위해 pad_token_id로 채웁니다.
        padded = new_item + [????] * (batch_max_length - len(new_item))
        
        # 힌트: 다음 토큰 예측에서는 inputs=padded[:-1], targets=padded[1:] 형태를 만듭니다.
        inputs = torch.tensor(padded[:????]) 
    
        # 정답은 첫 번째 토큰 제외 (1 ~ n) -> 다음 토큰 예측 문제이므로
        targets = torch.tensor(padded[????:]) 

        # 중요: 패딩 부분 마스킹 처리
        # 타겟에서 패딩 토큰인 부분은 ignore_index(-100)로 바꿔서 CrossEntropyLoss 계산 시 무시되게 함
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        
        # 패딩이 시작되는 지점 이후의 모든 타겟 값을 -100으로 설정
        if indices.numel() > 1:
            targets[indices[???:]] = ignore_index
        # endregion
        
        # 선택적으로 최대 시퀀스 길이 제한 (메모리 관리 등 목적)
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # 텐서로 변환 및 디바이스(GPU/CPU)로 이동
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

    #######################################
    # 3. 모델 미세 조정 (Finetuning)
    #######################################
    print("Initial losses")
    # 학습 전 초기 손실값 확인
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

    print("   Training loss:", train_loss)
    print("   Validation loss:", val_loss)

    start_time = time.time()
    # 옵티마이저 설정 (AdamW)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

    num_epochs = 2

    torch.manual_seed(123)
    
    # 실제 학습 수행 (previous_chapters.py에 정의된 함수 사용)
    train_losses, val_losses, tokens_seen = train_model_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=num_epochs, eval_freq=5, eval_iter=5,
        start_context=format_input(val_data[0]), tokenizer=tokenizer
    )

In [ ]:
## with DPO
# ==========================================
# 2. 데이터셋 및 Collate 함수 (DPO용)
# DPO는 (프롬프트, 선택된 답변, 거부된 답변)의 쌍이 필요합니다.
# ==========================================
class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        
        # 데이터를 미리 토큰화(Tokenization)하여 저장합니다.
        for entry in data:
            prompt = format_input(entry)
            rejected_response = entry["rejected"] # 👎 덜 선호되는 답변
            chosen_response = entry["chosen"]     # 👍 더 선호되는 답변

            prompt_tokens = tokenizer.encode(prompt)
            chosen_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{????}")
            rejected_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{????}")

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [ ]:
def custom_collate_fn(batch, pad_token_id=50256, allowed_max_length=None, mask_prompt_tokens=True, device="cpu"):
    """
    배치(Batch) 내의 데이터 길이를 맞추고(Padding), 마스크(Mask)를 생성하는 함수입니다.
    """
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [], # Loss 계산 시 무시할 부분 (패딩 등)
        "chosen_mask": []
    }

    # 1. 배치 내에서 가장 긴 시퀀스 길이 찾기 (패딩을 위해)
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key]) + 1 for item in batch)
            max_length_common = max(max_length_common, current_max)

    # 2. 패딩 추가 및 마스크 생성
    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            sequence = item[key]
            # 최대 길이에 맞춰 패딩 토큰 추가
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            
            # 기본 마스크: 데이터가 있는 곳은 1(True), 패딩은 0(False)
            mask = torch.ones(len(padded)).bool()
            mask[len(sequence):] = False  # 패딩 부분 마스킹

            # [중요] 프롬프트 부분 마스킹 (선택적)
            # DPO는 '답변'의 확률 차이를 학습하므로, 질문(Prompt) 부분은 Loss 계산에서 제외합니다.
            # +2는 \n\n때문
            if mask_prompt_tokens:
                mask[:prompt.shape[0] + 2] = False 

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    # 3. 텐서 변환 및 디바이스(GPU/CPU) 이동
    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        tensor_stack = torch.stack(batch_data[key])
        # 길이 제한이 있다면 자르기
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]
        batch_data[key] = tensor_stack.to(device)

    return batch_data

# ==========================================
# 3. DPO Loss 및 Log Probability 계산 함수
# DPO의 핵심 수학적 로직이 들어있는 부분입니다.
# ==========================================
def compute_logprobs(logits, labels, selection_mask=None):
    """
    모델의 출력(logits)과 정답(labels)을 받아 해당 정답 토큰의 로그 확률을 계산합니다.
    """
    # Auto-regressive 모델 특성상, 입력 [A, B, C]에 대해 예측은 [B, C, D]가 되므로 시프트(Shift)합니다.
    # TODO: 자동회귀 시프트 인덱스를 채우세요.
    # 힌트: labels는 1칸 앞당기고 logits는 마지막 시점을 제외해 정렬합니다.
    labels = labels[:, ????:].clone() # labels는 b * token_sequence
    logits = logits[:, :????, :]      # logits는 b * token_sequence * vocab_dict
    
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 실제 정답 레이블에 해당하는 확률값만 추출 (gather 사용)
    # TODO: gather의 input 텐서를 채우세요.
    # 힌트: raw logits가 아닌 log_probs에서 정답 위치를 gather해야 합니다.
    selected_log_probs = torch.gather(
        input=????,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    if selection_mask is not None:
        # 마스크도 시프트하여 적용 (패딩이나 프롬프트 영역 무시)
        mask = selection_mask[:, 1:].clone()
        selected_log_probs = selected_log_probs * mask
        
        # 유효한 토큰들의 로그 확률 평균 계산
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)
        return avg_log_prob
    else:
        return selected_log_probs.mean(-1)

def compute_dpo_loss(model_chosen_logprobs, model_rejected_logprobs, 
                     reference_chosen_logprobs, reference_rejected_logprobs, beta=0.1):
    """
    DPO 손실 함수 계산:
    Policy 모델이 Reference 모델보다 'chosen' 답변을 더 선호하고, 'rejected' 답변을 덜 선호하도록 유도합니다.
    
    beta: Reference 모델에서 얼마나 벗어날지 제어하는 하이퍼파라미터 (보통 0.1~0.5)
    """

    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs
    
    logits = model_logratios - reference_logratios
    
    losses = -F.???(beta * logits)
    
    # 학습 추적용 보상(Reward) 계산 (실제 학습엔 안 쓰이고 로깅용)
    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()
    
    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()


def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):
    """
    하나의 배치에 대해 전체 DPO 과정을 수행하는 헬퍼 함수
    """
    # 1. 학습 중인 모델(Policy Model)의 로그 확률 계산 (기울기 계산 O)
    policy_chosen_log_probas = compute_logprobs(
        logits=policy_model(batch["chosen"]),
        labels=batch["chosen"],
        selection_mask=batch["chosen_mask"]
    )
    policy_rejected_log_probas = compute_logprobs(
        logits=policy_model(batch["rejected"]),
        labels=batch["rejected"],
        selection_mask=batch["rejected_mask"]
    )
    
    # 2. 기준 모델(Reference Model)의 로그 확률 계산 (기울기 계산 X -> 메모리 절약)
    with torch.no_grad():
        ref_chosen_log_probas = compute_logprobs(
            logits=reference_model(batch["chosen"]),
            labels=batch["chosen"],
            selection_mask=batch["chosen_mask"]
        )
        ref_rejected_log_probas = compute_logprobs(
            logits=reference_model(batch["rejected"]),
            labels=batch["rejected"],
            selection_mask=batch["rejected_mask"]
        )

    # 3. 최종 Loss 계산
    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_logprobs=policy_chosen_log_probas,
        model_rejected_logprobs=policy_rejected_log_probas,
        reference_chosen_logprobs=ref_chosen_log_probas,
        reference_rejected_logprobs=ref_rejected_log_probas,
        beta=beta
    )
    return loss, chosen_rewards, rejected_rewards
